# UCI BNN — Sampler Investigation

Loads **all available samplers** for one `(dataset, split)` and compares them across:
metrics, sparsity, calibration, posterior noise, ESS, and predictive intervals.

In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
from torch import Tensor
from torch.distributions import Normal

plt.rcParams.update({
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.3,
    "font.size":          11,
})

import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

## 1. Choose (dataset, split)

In [ ]:
RESULTS_DIR   = Path("results/uci_bnn")

DATASET       = "boston"
SPLIT_ID      = 0
LEARNED_NOISE = True   # True -> *_ln.pt only;  False -> fixed-noise only

# Display-name overrides (stem -> label)
LABELS = {
    "zigzag":              "ZigZag",
    "sticky_zigzag":       "Sticky ZigZag",
    "boomerang":           "Boomerang",
    "sticky_boomerang":    "Sticky Boomerang",
    "nuts":                "NUTS",
    "nuts_horseshoe":      "NUTS-HS",
    "zigzag_ln":           "ZigZag (LN)",
    "sticky_zigzag_ln":    "Sticky ZigZag (LN)",
    "boomerang_ln":        "Boomerang (LN)",
    "sticky_boomerang_ln": "Sticky Boomerang (LN)",
    "nuts_ln":             "NUTS (LN)",
    "nuts_horseshoe_ln":    "NUTS-HS (LN)",
}

# Colour palette — one colour per base sampler family
COLORS = {
    "zigzag":              "#00358B",
    "sticky_zigzag":       "#00358B",
    "boomerang":           "#FA5C00",
    "sticky_boomerang":    "#FA5C00",
    "nuts":                "#0CCA38",
    "nuts_horseshoe":      "#0CCA38",
    "zigzag_ln":           "#00358B",
    "sticky_zigzag_ln":    "#00358B",
    "boomerang_ln":        "#FA5C00",
    "sticky_boomerang_ln": "#FA5C00",
    "nuts_ln":             "#0CCA38",
    "nuts_horseshoe_ln":   "#0CCA38",
}

# Linestyle: solid for sticky/HS, dashed for vanilla
LINESTYLES = {
    "zigzag":              "--",
    "sticky_zigzag":       "-",
    "boomerang":           "--",
    "sticky_boomerang":    "-",
    "nuts":                "--",
    "nuts_horseshoe":      "-",
    "zigzag_ln":           "--",
    "sticky_zigzag_ln":    "-",
    "boomerang_ln":        "--",
    "sticky_boomerang_ln": "-",
    "nuts_ln":             "--",
    "nuts_horseshoe_ln":   "-",
}

## 2. Load runs and reconstruct data split

In [ ]:
from sazz.scripts.bnns.uci_bnn import (
    load_raw_datasets, make_split, build_target, build_target_learned_noise,
    BNNConfig, BASE_SEED,
)

split_dir = RESULTS_DIR / DATASET / f"split_{SPLIT_ID:02d}"
all_pts   = sorted(split_dir.glob("*.pt"))
print(split_dir)
# Filter to the chosen noise model
pt_files = [p for p in all_pts if p.stem.endswith("_ln") == LEARNED_NOISE]
assert pt_files, (
    f"No {'learned' if LEARNED_NOISE else 'fixed'}-noise .pt files found in {split_dir}.\n"
    f"Available: {[p.stem for p in all_pts]}"
)

noise_label = "learned noise" if LEARNED_NOISE else "fixed noise"
print(f"Noise model : {noise_label}")
print(f"Found {len(pt_files)} run(s) in {split_dir}:")
for p in pt_files:
    print(f"  {p.stem}")

In [ ]:
print("Loading raw dataset for test-set reconstruction...")
raw = load_raw_datasets()
X_all, y_all = raw[DATASET]

data   = make_split(X_all, y_all, seed=BASE_SEED + SPLIT_ID)
X_test = data["X_test"]
y_test = data["y_test"]
y_std  = data["y_std"]
print(f"  X_test: {X_test.shape}   y_std: {y_std:.4f}")

## 3. Rebuild targets and compute predictions

One `TorchTarget` is built per noise model type (fixed vs learned). The model reconstruction
is identical to `build_target` / `build_target_learned_noise` in `uci_bnn.py`.

In [ ]:
target_cache: dict[bool, object] = {}   # keyed by learned_noise bool

def get_target(run: dict, learned: bool):
    if learned not in target_cache:
        cfg = BNNConfig(
            layer_sizes = run["layer_sizes"],
            activation  = run["activation"],
            noise_std   = run["noise_std"],
            learned_noise = learned,
        )
        builder = build_target_learned_noise if learned else build_target
        print(f"  Building {'learned-noise' if learned else 'fixed-noise'} target "
              f"(layer_sizes={run['layer_sizes']}, act={run['activation']})...")
        target_cache[learned] = builder(data, cfg)
        print(f"    D = {target_cache[learned].D}")
    return target_cache[learned]


def predict_all(samples: Tensor, likelihood, X_new: Tensor) -> Tensor:
    return torch.stack([
        likelihood.predict(beta, X_new).squeeze(-1) for beta in samples
    ])  # [S, N]


runs: dict[str, dict] = {}   # stem -> payload

for pt in pt_files:
    stem    = pt.stem
    learned = stem.endswith("_ln")
    run     = torch.load(pt, map_location="cpu", weights_only=False)
    target  = get_target(run, learned)
    samples = run["samples"]   # [S, D]

    likelihood  = target.meta["model"].likelihood
    preds       = predict_all(samples, likelihood, X_test)   # [S, N]
    mean_pred   = preds.mean(0)
    epist_std   = preds.std(0)
    
    if learned:
        noise_samples = samples[:, -1].exp()       # [S]
        noise_std_eff = float(noise_samples.mean())
    else:
        noise_samples = None
        noise_std_eff = run["noise_std"]

    total_std = (epist_std ** 2 + noise_std_eff ** 2).sqrt()

    base_spec = target.meta.get("base_spec", target.meta["spec"])
    weight_samples = samples[:, :base_spec.D]

    runs[stem] = dict(
        run           = run,
        samples       = samples,
        weight_samples= weight_samples,
        base_spec     = base_spec,
        mean_pred     = mean_pred,
        epist_std     = epist_std,
        total_std     = total_std,
        noise_std_eff = noise_std_eff,
        noise_samples = noise_samples,
        learned       = learned,
        label         = LABELS.get(stem, stem),
        color         = COLORS.get(stem, "grey"),
        ls            = LINESTYLES.get(stem, "-"),
    )
    print(f"  [{stem}] done — {samples.shape[0]} samples")

sampler_order = list(runs.keys())

## 4. Metrics table

In [ ]:
from sazz.utils.metrics import ess_per_coord
import pandas as pd

def compute_rmse(y_true, mean_pred, y_std):
    return float(((mean_pred - y_true) ** 2).mean().sqrt()) * y_std

def compute_nll(y_true, mean_pred, total_std, y_std):
    ll = (-0.5 * ((y_true - mean_pred) / total_std) ** 2
          - total_std.log() - 0.5 * math.log(2 * math.pi)).mean()
    return float(-ll + math.log(y_std))

def compute_crps(y_true, mean_pred, total_std, y_std):
    d = Normal(0.0, 1.0)
    sigma = total_std * y_std
    z = (y_true * y_std - mean_pred * y_std) / sigma
    return float((sigma * (z * (2*d.cdf(z) - 1) + 2*d.log_prob(z).exp()
                           - 1/math.sqrt(math.pi))).mean())

def compute_coverage(y_true, mean_pred, total_std, level=0.9):
    z = Normal(0.0, 1.0).icdf(torch.tensor(0.5 + level / 2))
    return float(((y_true - mean_pred).abs() <= z * total_std).float().mean())

rows = []
for stem, s in runs.items():
    ess   = ess_per_coord(s["weight_samples"])
    elapsed = s["run"]["elapsed_sec"]
    rows.append({
        "Sampler":    s["label"],
        "RMSE":       compute_rmse(y_test, s["mean_pred"], y_std),
        "NLL":        compute_nll(y_test, s["mean_pred"], s["total_std"], y_std),
        "CRPS":       compute_crps(y_test, s["mean_pred"], s["total_std"], y_std),
        "Cov 90%":    compute_coverage(y_test, s["mean_pred"], s["total_std"], 0.90),
        "Cov 95%":    compute_coverage(y_test, s["mean_pred"], s["total_std"], 0.95),
        "ESS min":    float(ess.min()),
        "ESS/s":      float(ess.min()) / elapsed,
        "Time (s)":   elapsed,
    })

metrics_df = pd.DataFrame(rows).set_index("Sampler")
metrics_df.style.highlight_min(subset=["RMSE","NLL","CRPS"], color="#c6efce") \
                .highlight_max(subset=["ESS/s","Cov 90%","Cov 95%"], color="#c6efce") \
                .format(precision=3)

## 5. Metrics bar chart

In [ ]:
metric_cols = ["RMSE", "NLL", "CRPS"]
n_samplers  = len(sampler_order)
x           = np.arange(n_samplers)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col in zip(axes, metric_cols):
    vals   = [metrics_df.loc[runs[s]["label"], col] for s in sampler_order]
    colors = [runs[s]["color"] for s in sampler_order]
    bars   = ax.bar(x, vals, color=colors, edgecolor="white", linewidth=0.8)
    # hatch sticky ones
    for bar, stem in zip(bars, sampler_order):
        if "sticky" in stem or "horseshoe" in stem:
            bar.set_hatch("///")
    ax.set_xticks(x)
    ax.set_xticklabels([runs[s]["label"] for s in sampler_order],
                       rotation=35, ha="right", fontsize=9)
    ax.set_title(col)
    ax.set_ylabel(col)

fig.suptitle(f"{DATASET.capitalize()} — split {SPLIT_ID}", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Calibration curves

In [ ]:
levels = torch.linspace(0.02, 0.98, 40)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Ideal", zorder=0)

for stem, s in runs.items():
    covs = [compute_coverage(y_test, s["mean_pred"], s["total_std"], float(lv))
            for lv in levels]
    ax.plot(levels.numpy(), covs,
            color=s["color"], ls=s["ls"], lw=1.8, label=s["label"])

ax.set_xlabel("Nominal coverage")
ax.set_ylabel("Empirical coverage")
ax.set_title(f"{DATASET.capitalize()} — calibration")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

## 7. Sparsity profiles

In [ ]:
FREEZE_THR = 1e-3

# Use the first run's base_spec for the coordinate layout (all runs share it)
ref_spec = next(iter(runs.values()))["base_spec"]
D = ref_spec.D

fig, axes = plt.subplots(n_samplers, 1, figsize=(11, 2.2 * n_samplers), sharex=True)
if n_samplers == 1:
    axes = [axes]

# vertical lines marking layer boundaries
boundaries = []
offset = 0
for numel in ref_spec.numels[:-1]:
    offset += numel
    boundaries.append(offset)

for ax, stem in zip(axes, sampler_order):
    s   = runs[stem]
    fnz = (s["weight_samples"].abs() < FREEZE_THR).float().mean(0).numpy()
    ax.bar(range(D), fnz, width=1.0, linewidth=0, color=s["color"], alpha=0.85)
    ax.axhline(0.5, color="red", lw=0.8, ls="--")
    for b in boundaries:
        ax.axvline(b, color="black", lw=0.6, ls=":")
    ax.set_ylim(0, 1)
    ax.set_ylabel(s["label"], fontsize=8)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

axes[-1].set_xlabel("Parameter index")
fig.suptitle(f"{DATASET.capitalize()} — fraction of samples near zero (sparsity)",
             y=1.01)
plt.tight_layout()
plt.show()

## 8. Posterior noise (learned-noise runs)

In [ ]:
ln_runs = {k: v for k, v in runs.items() if v["learned"]}

if ln_runs:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for stem, s in ln_runs.items():
        ns = s["noise_samples"].numpy()
        ax.hist(ns, bins=60, density=True, alpha=0.55,
                color=s["color"], label=s["label"],
                histtype="stepfilled", edgecolor="none")
        ax.axvline(ns.mean(), color=s["color"], lw=1.8, ls=s["ls"])
    ax.set_xlabel(r"$\sigma$ (standardised scale)")
    ax.set_ylabel("Density")
    ax.set_title(f"{DATASET.capitalize()} — posterior noise")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No learned-noise runs in this split.")

## 9. ESS per coordinate

In [ ]:
fig, axes = plt.subplots(n_samplers, 1, figsize=(11, 2.2 * n_samplers), sharex=True)
if n_samplers == 1:
    axes = [axes]

for ax, stem in zip(axes, sampler_order):
    s   = runs[stem]
    ess = ess_per_coord(s["weight_samples"]).numpy()
    ax.bar(range(D), ess, width=1.0, linewidth=0, color=s["color"], alpha=0.85)
    for b in boundaries:
        ax.axvline(b, color="black", lw=0.6, ls=":")
    ax.set_ylabel(s["label"], fontsize=8)
    elapsed = s["run"]["elapsed_sec"]
    ax.set_title(f"{s['label']}   min ESS={ess.min():.0f}   "
                 f"ESS/s={ess.min()/elapsed:.1f}", fontsize=9, loc="left")

axes[-1].set_xlabel("Parameter index")
fig.suptitle(f"{DATASET.capitalize()} — ESS per coordinate", y=1.01)
plt.tight_layout()
plt.show()

## 10. Predictive intervals — all samplers

In [ ]:
yt_orig = y_test.numpy() * y_std
order   = np.argsort(yt_orig)
x_idx   = np.arange(len(order))

ncols = 2
nrows = math.ceil(n_samplers / ncols)
fig, axes = plt.subplots(nrows, ncols,
                         figsize=(7 * ncols, 3.5 * nrows),
                         sharey=True, sharex=True)
axes_flat = np.array(axes).flatten()

for ax, stem in zip(axes_flat, sampler_order):
    s        = runs[stem]
    mu_orig  = s["mean_pred"].numpy()[order] * y_std
    err_orig = s["total_std"].numpy()[order] * y_std

    ax.fill_between(x_idx, mu_orig - 1.96*err_orig, mu_orig + 1.96*err_orig,
                    alpha=0.25, color=s["color"], label="95% PI")
    ax.plot(x_idx, mu_orig, color=s["color"], lw=1.2, label="Mean pred.")
    ax.scatter(x_idx, yt_orig[order], s=6, color="black", zorder=3,
               alpha=0.6, label="Observed")
    cov90 = compute_coverage(y_test, s["mean_pred"], s["total_std"], 0.90)
    ax.set_title(f"{s['label']}  (90% cov = {cov90:.2f})", fontsize=9)
    ax.set_xlabel("Test point (sorted by y)")
    ax.set_ylabel("y (original scale)")

# hide empty subplots
for ax in axes_flat[n_samplers:]:
    ax.set_visible(False)

handles, labels_ = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=3,
           bbox_to_anchor=(0.5, -0.02), fontsize=9)
fig.suptitle(f"{DATASET.capitalize()} split {SPLIT_ID} — predictive intervals",
             fontsize=13)
plt.tight_layout()
plt.show()

## 11. Epistemic vs aleatoric uncertainty

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Epistemic (function) std, sorted by observed y ---
ax = axes[0]
for stem, s in runs.items():
    epist = s["epist_std"].numpy()[order] * y_std
    ax.plot(x_idx, epist, color=s["color"], ls=s["ls"], lw=1.4,
            label=s["label"], alpha=0.9)
ax.set_xlabel("Test point (sorted by y)")
ax.set_ylabel("Epistemic std (original scale)")
ax.set_title("Epistemic uncertainty")
ax.legend(fontsize=8)

# --- Total predictive std ---
ax = axes[1]
for stem, s in runs.items():
    total = s["total_std"].numpy()[order] * y_std
    ax.plot(x_idx, total, color=s["color"], ls=s["ls"], lw=1.4,
            label=s["label"], alpha=0.9)
ax.set_xlabel("Test point (sorted by y)")
ax.set_ylabel("Total predictive std (original scale)")
ax.set_title("Total uncertainty")
ax.legend(fontsize=8)

fig.suptitle(f"{DATASET.capitalize()} split {SPLIT_ID}", fontsize=13)
plt.tight_layout()
plt.show()